# Marine Monitoring Measurements from Basque Country Estuaries and Coast (1995–2023) Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset (long-term marine monitoring data) using the [`mlcroissant`](https://mlcroissant.org/) library, referencing all entities by their Croissant `@id`.

### Dataset Source
The dataset source is defined by a Croissant schema, accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.a32r-qw1w/fair2.json`

In [ ]:
# Install the mlcroissant library
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.a32r-qw1w/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("\nDescription:")
print(metadata.description)
print("\nLicense:", metadata.license)
print("Publication Date:", metadata.datePublished)


## 2. Data Overview
Review the available record sets, fields, and their Croissant `@id`s.

We will list all available record sets and sample a few records using their `@id`. The `mlcroissant` library provides full traceability via these IDs.

In [ ]:
# List all record sets by @id and label
record_sets = dataset.record_sets

record_sets_info = []
print("Available Record Sets:")
for rs in record_sets:
    rs_id = rs['@id']
    name = rs.get('name', rs_id)
    print(f"- {rs_id} (name: {name})")
    record_sets_info.append((rs_id, name))

# Display details for fields in each record set
for rs in record_sets[:2]:  # Display the first two record sets as an example
    print(f"\nFields for record set {rs['@id']}:")
    for field in rs.get('field', []):
        field_id = field['@id']
        field_name = field.get('name', field_id)
        print(f"  - Field @id: {field_id}, name: {field_name}, type: {field.get('dataType', 'Unknown')}")

# Sample 2 records from the first record set (if any)
if record_sets:
    sample_recordset_id = record_sets[0]['@id']
    print(f"\nSample records from record set {sample_recordset_id}:")
    for i, rec in enumerate(dataset.records(record_set=sample_recordset_id)):
        if i >= 2: break
        print(rec)


## 3. Data Extraction
Load complete data from a specific record set into a pandas DataFrame for further analysis.
We use the record set and field Croissant `@id` found in the previous step.

In [ ]:
# List all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load data for each record set
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:  # Only store non-empty record sets
        dataframes[rs_id] = pd.DataFrame(records)

# Show loaded DataFrame columns for the first non-empty record set
for rs_id in dataframes:
    print(f"Loaded DataFrame for record set @id: {rs_id}")
    print("Columns:", dataframes[rs_id].columns.tolist())
    display(dataframes[rs_id].head())
    sample_recordset_id = rs_id  # Select for further EDA
    break


## 4. Exploratory Data Analysis (EDA)

Apply common data processing and analysis operations to the data: filtering, normalization, and grouping.

We reference fields by their Croissant `@id`. Please update the `numeric_field_id` and `group_field_id` as appropriate for your analysis by inspecting the DataFrame columns above.

In [ ]:
# Identify a numeric field @id and a grouping field @id for EDA
# (Adjust these IDs based on actual DataFrame columns you see above)

numeric_field_id = None
group_field_id = None
for col in dataframes[sample_recordset_id].columns:
    # Heuristic: select first numeric-looking column
    if pd.api.types.is_numeric_dtype(dataframes[sample_recordset_id][col]):
        numeric_field_id = col
        break

# Example: choose a grouping field (likely location or station)
for col in dataframes[sample_recordset_id].columns:
    if "station" in col.lower() or "location" in col.lower() or "site" in col.lower():
        group_field_id = col
        break

print(f"Numeric field selected: {numeric_field_id}")
print(f"Grouping field selected: {group_field_id if group_field_id else 'None found'}")

df = dataframes[sample_recordset_id]

if numeric_field_id is not None:
    threshold = df[numeric_field_id].quantile(0.90)  # Example: filter to top 10% values
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize the numeric field (mean 0, std 1)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by the group field and calculate the mean
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} by {group_field_id} (for outlier records):")
        display(grouped_df.head())
else:
    print("No numeric field found in the selected record set for EDA.")


## 5. Visualization
Visualize the numeric variable's distribution and its variation across groups using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=40, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        top_groups = df[group_field_id].value_counts().index[:8]
        plt.figure(figsize=(10,6))
        sns.boxplot(
            data=df[df[group_field_id].isin(top_groups)],
            x=group_field_id, y=numeric_field_id
        )
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id} (top groups)")
        plt.show()
else:
    print("No suitable numeric field available for visualization.")


## 6. Conclusion
In this notebook, we have:
- Loaded a rich environmental monitoring dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`.
- Explored the metadata, available record sets, and fields.
- Extracted tabular data into pandas DataFrames and performed data filtering, normalization, and grouping.
- Visualized the distribution and group-wise patterns of the variables.

This approach using Croissant `@id` enables robust, reproducible, and semantically clear data exploration for scientific, ecological, and machine learning workflows.

You can now proceed with further analyses, feature engineering, or modeling steps tailored to your specific objectives.